In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().resolve()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

universe_500 = pd.read_csv(DATA_RAW / "universe_500.csv")["ticker"].astype(str).tolist()
fund_500 = pd.read_csv(DATA_RAW / "fundamentals_500_raw.csv")
ret_500 = pd.read_csv(DATA_PROCESSED / "return_panel_500.csv", parse_dates=["date"])
mom_500 = pd.read_csv(DATA_PROCESSED / "momentum_panel_500.csv", parse_dates=["date"])

fund_500["report_date"] = pd.to_datetime(fund_500["report_date"])
fund_500["ticker"] = fund_500["ticker"].astype(str)
ret_500["ticker"] = ret_500["ticker"].astype(str)
mom_500["ticker"] = mom_500["ticker"].astype(str)

covered_tickers = sorted(
    set(fund_500["ticker"].dropna().unique())
    .intersection(set(ret_500["ticker"].dropna().unique()))
    .intersection(set(mom_500["ticker"].dropna().unique()))
)

fund_cov = fund_500[fund_500["ticker"].isin(covered_tickers)].copy()
ret_cov = ret_500[ret_500["ticker"].isin(covered_tickers)].copy()
mom_cov = mom_500[mom_500["ticker"].isin(covered_tickers)].copy()

fund_cov["book_to_sales"] = fund_cov["book_value"] / fund_cov["revenue"].replace(0, np.nan)
fund_cov["gross_profit_margin"] = (fund_cov["revenue"] - fund_cov["cogs"]) / fund_cov["revenue"].replace(0, np.nan)
fund_cov["net_margin"] = fund_cov["net_income"] / fund_cov["revenue"].replace(0, np.nan)
fund_cov["roe"] = fund_cov["net_income"] / fund_cov["book_value"].replace(0, np.nan)
fund_cov["assets_to_equity"] = fund_cov["total_assets"] / fund_cov["book_value"].replace(0, np.nan)
fund_cov = fund_cov.replace([np.inf, -np.inf], np.nan)

def zscore(s):
    sd = s.std(ddof=0)
    return (s - s.mean()) / sd if pd.notna(sd) and sd != 0 else np.nan

for c in ["book_to_sales", "gross_profit_margin", "net_margin", "roe", "assets_to_equity"]:
    fund_cov[c + "_z"] = fund_cov.groupby("report_date")[c].transform(zscore)

value_cov = fund_cov.groupby(["ticker", "report_date"], as_index=False).agg(
    book_to_sales=("book_to_sales", "mean"),
    gross_profit_margin=("gross_profit_margin", "mean"),
    net_margin=("net_margin", "mean"),
    roe=("roe", "mean"),
    assets_to_equity=("assets_to_equity", "mean")
)
value_cov["value_score"] = value_cov.groupby("report_date")["book_to_sales"].transform(zscore)

quality_cov = fund_cov.groupby(["ticker", "report_date"], as_index=False).agg(
    book_to_sales=("book_to_sales_z", "mean"),
    gross_profit_margin=("gross_profit_margin_z", "mean"),
    net_margin=("net_margin_z", "mean"),
    roe=("roe_z", "mean"),
    assets_to_equity=("assets_to_equity_z", "mean")
)
quality_cov["quality_score"] = quality_cov[["gross_profit_margin", "net_margin", "roe", "assets_to_equity", "book_to_sales"]].mean(axis=1)

fund_cov["eps"] = fund_cov["net_income"] / fund_cov["shares_diluted"].replace(0, np.nan)
fund_cov["eps_lag4"] = fund_cov.groupby("ticker")["eps"].shift(4)
fund_cov["sue"] = fund_cov["eps"] - fund_cov["eps_lag4"]
fund_cov["sue_z"] = fund_cov.groupby("report_date")["sue"].transform(zscore)
sue_cov = fund_cov[["ticker", "report_date", "sue", "sue_z"]].copy()

mom_cov["momentum_12m_z"] = mom_cov.groupby("date")["momentum_12m"].transform(zscore)

value_cov.to_csv(OUT / "factor_C_value_overlap.csv", index=False)
quality_cov.to_csv(OUT / "factor_C_quality_overlap.csv", index=False)
sue_cov.to_csv(OUT / "factor_C_sue_overlap.csv", index=False)
mom_cov.to_csv(OUT / "factor_C_momentum_overlap.csv", index=False)

coverage = pd.DataFrame({
    "model": ["C_price_plus_fundamentals"],
    "n_universe_500": [len(universe_500)],
    "n_covered_tickers": [len(covered_tickers)],
    "n_fund_rows": [len(fund_cov)],
    "n_return_rows": [len(ret_cov)],
    "n_momentum_rows": [len(mom_cov)]
})
coverage.to_csv(OUT / "factor_C_coverage.csv", index=False)

pd.DataFrame({"ticker": covered_tickers}).to_csv(DATA_RAW / "universe_overlap.csv", index=False)

print(coverage)
print(value_cov.head())
print(quality_cov.head())
print(sue_cov.head())

                       model  n_universe_500  n_covered_tickers  n_fund_rows  \
0  C_price_plus_fundamentals             500                358         6193   

   n_return_rows  n_momentum_rows  
0          52984            46875  
  ticker report_date  book_to_sales  gross_profit_margin  net_margin  \
0      A  2020-07-31       3.950040             1.469469    0.157811   
1      A  2020-10-31       3.285907             1.468645    0.149697   
2      A  2021-01-31       3.103359             1.458656    0.186047   
3      A  2021-04-30       3.154098             1.464262    0.141639   
4      A  2021-07-31       3.118537             1.462799    0.166456   

        roe  assets_to_equity  value_score  
0  0.039952          1.916483     0.679565  
1  0.045557          1.975580     0.512353  
2  0.059950          2.013739     0.329747  
3  0.044906          2.161746     0.385610  
4  0.053376          2.121108     0.341430  
  ticker report_date  book_to_sales  gross_profit_margin  net_ma